# Exploratory analysis

This notebook reads the cleaned UCI Online Retail transactions and the modeling table produced by `python -m src.pipeline`. Interpretations are generated from those computed tables. They are not hardcoded findings.

The dataset is the UCI Online Retail file (CC BY 4.0). It is not AstroTalk customer data.


In [ ]:
import json
from pathlib import Path

import pandas as pd

from src.config import get_settings
from src.eda import run_eda

settings = get_settings()
transactions = pd.read_csv(settings.processed_dir / "transactions_clean.csv", parse_dates=["invoice_date"])
modeling = pd.read_csv(settings.processed_dir / "modeling_table.csv", parse_dates=["snapshot_date"])
quality = json.loads((settings.reports_dir / "data_quality.json").read_text(encoding="utf-8"))
segments = pd.read_csv(settings.reports_dir / "segment_summary.csv")
cohort_path = settings.reports_dir / "sql" / "09_monthly_cohorts.csv"
cohorts = pd.read_csv(cohort_path) if cohort_path.exists() else pd.DataFrame()
summary = run_eda(transactions, modeling, quality, segments, cohorts, settings.reports_dir / "figures")
summary["purchase_customers"], summary["repeat_purchase_rate"], summary["test_positive_rate"]


## How to read each figure

For every chart the next cell prints four things: what it shows, the pattern measured in this file, why that pattern matters, and what would still need to be checked before a business decision.


In [ ]:
from IPython.display import Image, Markdown, display

for note in summary["interpretations"]:
    display(Markdown(
        f"### {note['chart']}\n\n"
        f"**What it shows.** {note['shows']}\n\n"
        f"**Pattern in this dataset.** {note['pattern']}\n\n"
        f"**Why it matters.** {note['why']}\n\n"
        f"**Further analysis before a decision.** {note['further']}"
    ))

for path in summary["plots"]:
    display(Markdown(f"**{Path(path).name}**"))
    display(Image(filename=path))
